# Importando as Bibliotecas

In [3]:
import numpy as np
import pandas as pd
import mygene

from pathlib import Path
from pydeseq2.dds import DeseqDataSet

## Definindo o caminho dos dados

In [4]:
# Diretório principal do projeto
PROJECT_DIR = Path.cwd().parent

# Diretórios de dados
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Criar diretório de saída, caso não exista
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório do projeto:", PROJECT_DIR)
print("Diretório dos dados originais:", RAW_DIR)
print("Diretório dos dados processados:", PROCESSED_DIR)

Diretório do projeto: c:\Users\willi\meu_projeto_python\Dissertação\RBras
Diretório dos dados originais: c:\Users\willi\meu_projeto_python\Dissertação\RBras\data\raw
Diretório dos dados processados: c:\Users\willi\meu_projeto_python\Dissertação\RBras\data\processed


## Importando Base de Dados

In [5]:
# Carregar tabela clínica principal
paciente_df = pd.read_csv(
    RAW_DIR / "pacientes.txt",
    comment="#",
    sep="\t"
)

# Carregar tabela contendo os subtipos moleculares
subtipos_df = pd.read_csv(
    RAW_DIR / "data_clinical_patient.txt",
    sep="\t",
    comment="#"
)

# Padronizar identificadores dos pacientes
paciente_df["PATIENT_ID"] = (
    paciente_df["PATIENT_ID"].astype(str).str.strip().str[:12]
)

subtipos_df["PATIENT_ID"] = (
    subtipos_df["PATIENT_ID"].astype(str).str.strip().str[:12]
)

# Remover duplicatas e pacientes sem subtipo definido
subtipos_df = subtipos_df.drop_duplicates(
    subset=["PATIENT_ID"]
)

subtipos_df = subtipos_df.dropna(
    subset=["SUBTYPE"]
)

# Integrar as tabelas clínicas
clinical_merged = pd.merge(
    paciente_df,
    subtipos_df[["PATIENT_ID", "SUBTYPE"]],
    on="PATIENT_ID",
    how="inner"
).set_index("PATIENT_ID")

print("Tabela clínica integrada:", clinical_merged.shape)

Tabela clínica integrada: (981, 31)


In [6]:
# Carregar matriz de contagens brutas
df_raw = pd.read_csv(
    RAW_DIR / "data_mrna_seq_read_counts.txt",
    sep="\t",
    index_col=0
)

# Selecionar apenas tumores primários (-01A)
colunas_01a = [
    c for c in df_raw.columns
    if str(c).endswith("-01A")
]

df_01a = df_raw[colunas_01a].copy()

# Padronizar identificadores para 12 caracteres
df_01a.columns = [
    str(c)[:12] for c in df_01a.columns
]

# Remover alíquotas duplicadas
df_01a = df_01a.loc[
    :, ~df_01a.columns.duplicated(keep="first")
]

# Identificar pacientes presentes nas duas bases
pacientes_comuns = df_01a.columns.intersection(
    clinical_merged.index
)

# Alinhar matriz de expressão e tabela clínica
df_counts = df_01a[pacientes_comuns].copy()

clinical_final = clinical_merged.loc[
    pacientes_comuns
].copy()

print(
    f"Coorte final: {df_counts.shape[1]} pacientes "
    f"x {df_counts.shape[0]} genes."
)

Coorte final: 970 pacientes x 40796 genes.


In [7]:
ids_pacientes_txt = set(paciente_df["PATIENT_ID"])

ids_subtipos_validos = set(
    subtipos_df["PATIENT_ID"]
)

ids_rnaseq_01a = set(df_01a.columns)

print("IDs em pacientes.txt:", len(ids_pacientes_txt))
print("IDs com PAM50 válido:", len(ids_subtipos_validos))
print("IDs com RNA-seq:", len(ids_rnaseq_01a))

intersecao = (
    ids_pacientes_txt
    & ids_subtipos_validos
    & ids_rnaseq_01a
)

print("Interseção total:", len(intersecao))

IDs em pacientes.txt: 1098
IDs com PAM50 válido: 981
IDs com RNA-seq: 1081
Interseção total: 970


## Mapeamento Genético por HUGO

In [8]:
print("Iniciando mapeamento dos identificadores dos genes...")

mg = mygene.MyGeneInfo()

gene_ids = df_counts.index.astype(str).tolist()

res = mg.querymany(
    gene_ids,
    scopes="entrezgene,ensembl.gene",
    fields="symbol",
    species="human"
)

dict_map = {}

for row in res:

    query_id = str(row["query"])

    dict_map[query_id] = (
        row["symbol"]
        if "symbol" in row and pd.notna(row["symbol"])
        else query_id
    )

# Aplicar mapeamento
df_mapped = df_counts.copy()

df_mapped.index = (
    df_mapped.index.astype(str).map(dict_map)
)

# Agrupar genes com o mesmo símbolo utilizando soma
df_symbols = df_mapped.groupby(
    df_mapped.index
).sum()

print(
    f"Mapeamento concluído: {df_symbols.shape[0]} genes únicos."
)

Iniciando mapeamento dos identificadores dos genes...


33 input query terms found dup hits:	[('118126072', 2), ('246122', 2), ('100101120', 2), ('100131060', 2), ('100506392', 2), ('100750325'
31 input query terms found no hit:	['100874233', '100874266', '100874283', '101929598', '102477328', '102724660', '103021294', '1053712


Mapeamento concluído: 40761 genes únicos.


## Filtragem de genes com baixa contagem

In [9]:
# Número de pacientes
n_pacientes = df_symbols.shape[1]

# Exigir expressão >= 10 em pelo menos 5% dos pacientes
min_samples = int(np.ceil(0.05 * n_pacientes))

print(
    f"Critério: contagem >= 10 em pelo menos "
    f"{min_samples} pacientes."
)

# Aplicar filtro
genes_para_manter = (
    (df_symbols >= 10).sum(axis=1) >= min_samples
)

df_filtered = df_symbols.loc[
    genes_para_manter
].copy()

print(
    f"Genes antes do filtro: {df_symbols.shape[0]}"
)

print(
    f"Genes após o filtro: {df_filtered.shape[0]}"
)

Critério: contagem >= 10 em pelo menos 49 pacientes.
Genes antes do filtro: 40761
Genes após o filtro: 22896


## Aplicação do VST

In [10]:
print("Iniciando transformação VST...")

# Transpor matriz: amostras x genes
df_counts_vst = df_filtered.T.astype(int)

# Instanciar objeto PyDESeq2
dds = DeseqDataSet(
    counts=df_counts_vst,
    metadata=clinical_final,
    design="~1"
)

# Aplicar VST
dds.vst()

# Extrair matriz transformada
df_vst = pd.DataFrame(
    dds.layers["vst_counts"],
    index=df_counts_vst.index,
    columns=df_counts_vst.columns
)

print("VST concluída!")
print("Dimensões da matriz:", df_vst.shape)

Iniciando transformação VST...
Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 3.33 seconds.

Fitting dispersions...
... done in 44.94 seconds.



VST concluída!
Dimensões da matriz: (970, 22896)


## Salvando base VST

In [11]:
# Salvar matriz VST
df_vst.to_csv(
    PROCESSED_DIR / "df_vst.csv"
)

# Salvar informações clínicas
clinical_final.to_csv(
    PROCESSED_DIR / "clinical_final.csv"
)

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


## Cálculo do MAD

In [12]:
from scipy.stats import median_abs_deviation
from sklearn.preprocessing import StandardScaler

# Calcular MAD individualmente para cada gene
mad_values = median_abs_deviation(
    df_vst.values,
    axis=0,
    scale=1,
    nan_policy="omit"
)

# Organizar os valores em uma Series
mad_series = pd.Series(
    mad_values,
    index=df_vst.columns,
    name="MAD"
)

# Ordenar genes do maior para o menor MAD
mad_series = mad_series.sort_values(
    ascending=False
)

print("Cálculo do MAD concluído.")
print(f"Total de genes avaliados: {len(mad_series)}")

display(mad_series.head(10))

Cálculo do MAD concluído.
Total de genes avaliados: 22896


Entrez_Gene_Id
SCGB2A2        3.986831
CYP2B7P        3.394594
GSTM1          3.380756
SCGB1D2        3.256226
TM4SF18-AS1    3.182218
PIP            3.054596
CYP4Z1         2.875923
TFAP2B         2.869857
HMGCS2         2.821156
PGR            2.792235
Name: MAD, dtype: float64

## Filtro por MAD

In [13]:
# Quantidades de genes que serão avaliadas
gene_counts = [3000, 5000, 10000, 20000]

# Dicionário para armazenar os conjuntos
datasets = {}

for n_genes in gene_counts:

    # Selecionar os genes com maiores valores de MAD
    selected_genes = mad_series.head(n_genes).index

    # Criar matriz correspondente
    df_selected = df_vst.loc[:, selected_genes].copy()

    # Armazenar conjunto sem Z-score
    datasets[f"VST_MAD_{n_genes}"] = df_selected

    print(
        f"{n_genes} genes selecionados: "
        f"{df_selected.shape}"
    )

3000 genes selecionados: (970, 3000)
5000 genes selecionados: (970, 5000)
10000 genes selecionados: (970, 10000)
20000 genes selecionados: (970, 20000)


## Aplicando Z-score nos genes

In [14]:
datasets_zscore = {}

for n_genes in gene_counts:

    # Recuperar conjunto sem Z-score
    df_selected = datasets[f"VST_MAD_{n_genes}"]

    # Padronizar cada gene (coluna)
    scaler = StandardScaler()

    zscore_values = scaler.fit_transform(df_selected)

    # Reconstruir DataFrame preservando índices e colunas
    df_zscore = pd.DataFrame(
        zscore_values,
        index=df_selected.index,
        columns=df_selected.columns
    )

    datasets_zscore[
        f"VST_ZSCORE_MAD_{n_genes}"
    ] = df_zscore

    print(
        f"Z-score aplicado em {n_genes} genes: "
        f"{df_zscore.shape}"
    )

Z-score aplicado em 3000 genes: (970, 3000)
Z-score aplicado em 5000 genes: (970, 5000)
Z-score aplicado em 10000 genes: (970, 10000)
Z-score aplicado em 20000 genes: (970, 20000)


## Salvando as bases filtrados por MAD

In [15]:
from pathlib import Path

DATASETS_DIR = PROCESSED_DIR / "mad_datasets"

DATASETS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Salvar conjuntos sem Z-score
for name, df in datasets.items():

    df.to_csv(
        DATASETS_DIR / f"{name}.csv"
    )

# Salvar conjuntos com Z-score
for name, df in datasets_zscore.items():

    df.to_csv(
        DATASETS_DIR / f"{name}.csv"
    )

print("Os oito conjuntos foram salvos com sucesso!")

Os oito conjuntos foram salvos com sucesso!


In [18]:
df = pd.read_csv(
    "../data/processed/mad_datasets/VST_MAD_5000.csv",
    index_col=0
)

print(df.shape)
display(df.iloc[:25, :10])

(970, 5000)


,SCGB2A2,CYP2B7P,GSTM1,SCGB1D2,TM4SF18-AS1,PIP,CYP4Z1,TFAP2B,HMGCS2,PGR
TCGA-3C-AAAU,17.467529,11.256350,11.163345,13.702970,8.206402,14.491205,6.291315,11.562620,4.201378,12.569464
TCGA-3C-AALI,17.644945,5.189737,6.858270,12.033904,11.643623,9.901424,11.713287,13.611586,7.793024,6.518027
TCGA-3C-AALJ,2.933732,11.386763,2.933732,4.284520,10.289572,2.933732,5.238104,3.732181,2.933732,10.419172
TCGA-3C-AALK,10.233443,10.871461,11.765203,8.072244,13.543867,13.048473,7.523470,12.673060,13.106527,12.589761
TCGA-4H-AAAK,16.512222,11.221688,11.741722,10.877150,6.042894,9.138891,8.453957,13.331718,7.680217,13.582350
TCGA-5L-AAT0,13.380735,6.816802,8.403923,10.714908,11.549530,9.547877,10.953905,13.242835,6.676549,10.932148
TCGA-5T-A9QA,18.194673,7.758514,4.197625,14.070395,14.307280,10.224365,2.933732,4.036585,5.858446,8.493372
TCGA-A1-A0SB,9.889253,4.552409,2.933732,9.430114,5.177770,7.171063,5.598756,7.217901,7.600029,5.318430
TCGA-A1-A0SD,11.626115,12.003050,2.933732,9.212591,11.080063,13.337070,12.008547,11.155789,12.117997,10.957824
TCGA-A1-A0SE,12.417866,7.984264,12.663811,11.480064,5.932582,8.914933,7.157649,12.639285,8.788211,14.277150


In [17]:
df = pd.read_csv(
    "../data/processed/mad_datasets/VST_ZSCORE_MAD_5000.csv",
    index_col=0
)

print(df.shape)
display(df.iloc[:5, :10])

(970, 5000)


,SCGB2A2,CYP2B7P,GSTM1,SCGB1D2,TM4SF18-AS1,PIP,CYP4Z1,TFAP2B,HMGCS2,PGR
TCGA-3C-AAAU,1.318581,0.416666,1.232266,1.108669,0.007591,1.121006,-0.630957,0.594545,-1.079134,0.602742
TCGA-3C-AALI,1.356174,-1.167601,0.046758,0.696016,0.932228,-0.053006,0.911325,1.167308,-0.048012,-1.262676
TCGA-3C-AALJ,-1.760937,0.450722,-1.033960,-1.219909,0.567978,-1.835259,-0.930544,-1.594360,-1.443060,-0.060108
TCGA-3C-AALK,-0.214225,0.316154,1.398002,-0.283448,1.443407,0.751973,-0.280470,0.904954,1.477434,0.608999
TCGA-4H-AAAK,1.116164,0.407614,1.391536,0.410025,-0.574409,-0.248053,-0.015793,1.089074,-0.080398,0.914975
